# PDF 规则文档解析、OCR、Milvus 入库与 Dify RAG 文件生成测试

这个 notebook 用于本地 review：从一个 PDF 文件开始，显式演示每个功能块。

流程：
1. 配置本地 PDF、Milvus、本地 BGE embedding、本地 PaddleOCR HTTP 服务。
2. 对扫描版 PDF 使用开源可本地部署 OCR（PaddleOCR）接口。
3. 按 `manual/policy/contract/faq/generic` 类型解析并分块。
4. 将分块写入 Milvus。
5. 将相同分块导出成 Dify RAG 需要的 JSONL segment 文件。
6. 可选：调用 Dify Knowledge API 创建文档或写入自定义 chunks。

> 说明：notebook 默认不会联网；所有服务地址都指向本地或内网。请先启动 Milvus、BGE embedding 服务和 OCR 服务。


## 0. 安装与路径准备

在仓库根目录运行 notebook 时，下面代码会把 `src/` 加到 `PYTHONPATH`，便于直接 import 当前代码。


In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

# TODO: 改成你本地要测试的 PDF。支持文本型 PDF；扫描型 PDF 需要配置 OCR_API_URL。
PDF_PATH = Path("./docs/manual.pdf").resolve()
OUTPUT_DIR = REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
DIFY_JSONL_PATH = OUTPUT_DIR / "manual.dify.jsonl"

print("PDF_PATH=", PDF_PATH)
print("DIFY_JSONL_PATH=", DIFY_JSONL_PATH)


## 1. 本地开源 OCR 接口（PaddleOCR）

OCR 选型建议：
- **PaddleOCR**：开源、中文效果好、可本地部署，适合扫描版制度/手册 PDF。
- 部署方式建议做成一个 HTTP 服务：输入 PDF 文件路径，输出每页文本和可选 metadata。

本项目的 `LocalOcrApiProvider` 默认请求：

```http
POST http://localhost:8001/ocr/pdf
Content-Type: application/json

{"path": "/absolute/path/manual.pdf"}
```

返回建议：

```json
{
  "pages": [
    {
      "page_number": 1,
      "text": "1 安全说明 ........ 1-1\n1.1 人员防护 ........ 1-1",
      "metadata": {"ocr_engine": "paddleocr", "ocr_confidence": 0.96}
    }
  ]
}
```

如果 PDF 本身可抽取文本，可以不配置 OCR；如果是扫描图片 PDF，则建议打开 OCR。


In [ ]:
from data_agent import LocalOcrApiProvider

OCR_API_URL = "http://localhost:8001/ocr/pdf"  # PaddleOCR 本地 HTTP 服务地址
USE_OCR = True  # 文本型 PDF 可设为 False

ocr_provider = LocalOcrApiProvider(OCR_API_URL) if USE_OCR else None
print("OCR enabled:", bool(ocr_provider))


## 2. 本地 BGE embedding 接口

embedding 使用本地 BGE HTTP API，默认 OpenAI-compatible 形式：

```http
POST http://localhost:8000/v1/embeddings
Content-Type: application/json

{"model": "bge-m3", "input": ["文本1", "文本2"]}
```

返回需要包含 `data[].embedding` 或 `embeddings`。


In [ ]:
from data_agent import LocalBGEEmbeddingProvider

BGE_API_URL = "http://localhost:8000/v1/embeddings"
BGE_MODEL = "bge-m3"
BGE_DIMENSION = 1024

embedding_provider = LocalBGEEmbeddingProvider(
    api_url=BGE_API_URL,
    model=BGE_MODEL,
    dimension=BGE_DIMENSION,
)
print("Embedding provider:", embedding_provider.model, embedding_provider.dimension)


## 3. PDF 解析与分块

`document_type` 控制切割策略：
- `manual`：产品手册/目录型 PDF，适合 `1`、`1.1`、`9.36.1` 这类章节。
- `policy`：制度/规章。
- `contract`：合同/协议。
- `faq`：问答型文档。
- `generic`：通用文本。

扫描版 PDF 会先尝试普通文本抽取；如果没有文本且配置了 `ocr_provider`，则自动走 OCR。


In [ ]:
from data_agent import RuleDocumentParser

parser = RuleDocumentParser(
    document_type="manual",
    ocr_provider=ocr_provider,
    ocr_on_empty=True,
)

chunks = parser.parse(
    PDF_PATH,
    document_format="pdf",
    extra_metadata={
        "department": "maintenance",
        "source_kind": "local_pdf_review",
    },
)

print("chunk_count=", len(chunks))
for chunk in chunks[:3]:
    print("---")
    print(chunk.metadata_payload())
    print(chunk.text[:300])


## 4. 目录页解析（可选）

对于截图中那种 `1.1 标题 ...... 1-1` 目录页，OCR 后可先抽目录结构，生成章节号、标题、页码标签、层级。目录结构可以作为 metadata 保存，也可以用于后续按章节路由。


In [ ]:
from data_agent import parse_toc_entries

# 如果前几页是目录，可把这些页的 OCR/抽取文本拼起来后解析。这里演示使用前 5 个 chunk 的文本。
toc_text = "\n".join(chunk.text for chunk in chunks[:5])
toc_entries = parse_toc_entries(toc_text)

print("toc_entry_count=", len(toc_entries))
for entry in toc_entries[:10]:
    print(entry)


## 5. 写入 Milvus 知识库

这一块会对 chunks 调本地 BGE embedding，并写入 Milvus。

请确认：
- Milvus 已启动。
- BGE embedding 服务已启动。
- `MILVUS_URI` 和 collection 名正确。


In [ ]:
from data_agent import MilvusRuleKnowledgeBase

MILVUS_URI = "http://localhost:19530"
MILVUS_TOKEN = None
COLLECTION_NAME = "rule_knowledge_base"

kb = MilvusRuleKnowledgeBase(
    uri=MILVUS_URI,
    token=MILVUS_TOKEN,
    collection_name=COLLECTION_NAME,
    embedding_provider=embedding_provider,
)

# 取消下一行注释即可真正入库。review 时可以先只检查 chunks。
# upserted = kb.upsert_chunks(chunks)
# print("upserted=", upserted)


## 6. 导出 Dify RAG 需要的 JSONL 文件

这个文件保留每个 chunk 的 `content`、`keywords` 和 metadata，便于 Dify 导入或用脚本调用 Dify Knowledge API。


In [ ]:
from data_agent import write_dify_jsonl, chunks_to_dify_segments

segments = chunks_to_dify_segments(chunks)
print("segment_count=", len(segments))
print(segments[0].to_jsonl_payload() if segments else "no segments")

written = write_dify_jsonl(chunks, DIFY_JSONL_PATH)
print("written=", written)
print("output=", DIFY_JSONL_PATH)


## 7. 可选：直接调用 Dify Knowledge API

如果你希望由 Dify 做 RAG 问答，可以直接调用 Dify Knowledge API。

两种常见方式：
1. `create_document_by_text`：把本工具切好的 chunks 合并为带分隔符的文本，让 Dify 建文档和索引。
2. `create_segments`：在 Dify 已有 document 下，写入本工具生成的 segments。

请把 API key 放在服务端或本地安全配置中，不要提交到代码仓库。


In [ ]:
from data_agent import DifyKnowledgeClient

DIFY_API_BASE_URL = "https://your-dify.example.com/v1"
DIFY_API_KEY = "replace-with-your-dify-knowledge-api-key"
DIFY_DATASET_ID = "replace-with-your-dataset-id"

dify_client = DifyKnowledgeClient(
    api_base_url=DIFY_API_BASE_URL,
    api_key=DIFY_API_KEY,
)

# 方式 1：创建文本型文档（默认注释，避免 review 时误调用外部服务）
# response = dify_client.create_document_by_text(
#     dataset_id=DIFY_DATASET_ID,
#     name=PDF_PATH.name,
#     text="\n\n".join(chunk.text for chunk in chunks),
# )
# print(response)

# 方式 2：已有 document_id 后写入 chunks
# DOCUMENT_ID = "replace-with-document-id"
# response = dify_client.create_segments(DIFY_DATASET_ID, DOCUMENT_ID, segments)
# print(response)


## 8. 本地检索验证（Milvus）

入库后可用同一个 BGE embedding provider 在 Milvus 中检索，返回 evidence，再把 evidence 交给 LLM 或 Dify 工作流。


In [ ]:
QUESTION = "焊前需要做哪些安全检查？"

# 取消注释进行真实 Milvus 检索。
# results = kb.search(QUESTION, limit=5, metadata_filter='metadata["document_type"] == "manual"')
# for result in results:
#     print(result.score, result.metadata)
#     print(result.text[:300])
